# Data distribution analysis

In [1]:
import pandas as pd
import altair as alt
import numpy as np
import os

alt.data_transformers.enable("vegafusion")

tracks_path = os.path.join('..', 'dataset', 'cleaned_tracks.csv')
artists_path = os.path.join('..', 'dataset', 'artists.csv')

df_tracks = pd.read_csv(tracks_path, sep=',')
df_artists = pd.read_csv(artists_path, sep=';')

## Missing data analysis

In [2]:
def prepare_missing_data(df, dataset_name):
    if df is None:
        return pd.DataFrame()
    
    total_rows = len(df)
    missing_counts = df.isnull().sum()
    present_counts = df.notnull().sum()
    
    df_missing = pd.DataFrame({
        'Column': missing_counts.index,
        'Count': missing_counts.values,
        'Status': 'Missing',
        'Dataset': dataset_name
    })
    
    df_present = pd.DataFrame({
        'Column': present_counts.index,
        'Count': present_counts.values,
        'Status': 'Present',
        'Dataset': dataset_name
    })
    
    combined = pd.concat([df_missing, df_present])
    
    combined['Percentage'] = combined['Count'] / total_rows
    
    return combined

viz_data_tracks = prepare_missing_data(df_tracks, 'Tracks')
viz_data_artists = prepare_missing_data(df_artists, 'Artists')
full_viz_data = pd.concat([viz_data_tracks, viz_data_artists])

base = alt.Chart(full_viz_data).encode(
    x=alt.X('Percentage', axis=alt.Axis(format='%'), title='Percentage'),
    y=alt.Y('Column', sort=alt.EncodingSortField(field='Percentage', op='min', order='ascending'), title=None),
    tooltip=[
        'Column', 
        'Status', 
        alt.Tooltip('Count', format=','), 
        alt.Tooltip('Percentage', format='.1%')
    ]
)

# Create the stacked bars
bars = base.mark_bar().encode(
    color=alt.Color(
        'Status', 
        scale=alt.Scale(domain=['Present', 'Missing'], range=['#4c78a8', '#e45756']),
        legend=alt.Legend(title="Data Status")
    ),
    order=alt.Order('Status', sort='ascending') # Ensure consistent stacking order
)

# Facet the chart into two columns (one for Tracks, one for Artists)
# We use resolve_scale(y='independent') so each chart shows its own columns
chart = bars.properties(
    width=400,
    height=600  # Taller height to accommodate many columns
).facet(
    column=alt.Column('Dataset', title='Dataset missing data', sort=['Tracks', 'Artists'])
).resolve_scale(
    y='independent' 
)

display(chart)

alt.FacetChart(...)

## Correlation Analysis of Tracks Data

This section performs a correlation analysis on the `cleaned_tracks.csv` dataset, which contains cleaned data from the previous analysis steps.

The focus will be on identifying relationships between numerical variables (audio features, lyrics statistics, track popularity).

In [3]:
import pandas as pd
import altair as alt
import numpy as np
import os

alt.data_transformers.enable("vegafusion")

DataTransformerRegistry.enable('vegafusion')

In [4]:
dataset_path = os.path.join('..', 'dataset', 'cleaned_tracks.csv')
artists_path = os.path.join('..', 'dataset', 'artists.csv')

df_tracks = pd.read_csv(tracks_path, sep=',')
df_artists = pd.read_csv(artists_path, sep=';')

We filter the dataframe to include only relevant numerical columns for correlation analysis (grouped by type for clarity).

In [5]:
numerical_cols = [
    #metadata
    'popularity', 
    'stats_pageviews',
    'duration_ms',
    
    #audio features
    'bpm', 
    'centroid', 
    'rolloff', 
    'flux', 
    'rms', 
    'zcr', 
    'flatness', 
    'spectral_complexity', 
    'pitch', 
    'loudness',
    
    #lyrics stats
    'n_sentences', 
    'n_tokens', 
    'tokens_per_sent', 
    'char_per_tok', 
    'lexical_density', 
    'avg_token_per_clause',
    'swear_IT',
    'swear_EN'
]

cols_to_use = [col for col in numerical_cols if col in df_tracks.columns]

df_corr = df_tracks[cols_to_use].copy()

#ensure all data is numeric (coercing errors if any remain)
for col in df_corr.columns:
    df_corr[col] = pd.to_numeric(df_corr[col], errors='coerce')

print(f"selected {len(df_corr.columns)} numerical columns for analysis.")

selected 21 numerical columns for analysis.


We use the Pearson correlation coefficient to measure linear relationships. To visualize this in Altair (which requires long-format data), we will also transform the matrix and mask the upper triangle.

In [6]:
#calculate standard correlation matrix
corr_matrix = df_corr.corr()

#display the top 5 positive correlations (excluding self-correlation)
print("Top 5 Positive Correlations:")
c = corr_matrix.abs()
s = c.unstack()
so = s.sort_values(kind="quicksort", ascending=False)
print(so[so < 1.0].head(10)) #top 5 pairs (duplicates appear twice)

Top 5 Positive Correlations:
rms          loudness       0.995586
loudness     rms            0.995586
rolloff      zcr            0.969082
zcr          rolloff        0.969082
n_tokens     n_sentences    0.868337
n_sentences  n_tokens       0.868337
zcr          centroid       0.864166
centroid     zcr            0.864166
             rolloff        0.775364
rolloff      centroid       0.775364
dtype: float64


We now plot the correlation matrix using Altair.

In [7]:
#prepare data for altair
corr_long = corr_matrix.stack().reset_index()
corr_long.columns = ['Variable 1', 'Variable 2', 'Correlation']

corr_long['Correlation_Label'] = corr_long['Correlation'].apply(lambda x: f"{x:.2f}")

base = alt.Chart(corr_long).encode(
    x=alt.X('Variable 2', title=None, sort=cols_to_use),
    y=alt.Y('Variable 1', title=None, sort=cols_to_use)
)

heatmap = base.mark_rect().encode(
    color=alt.Color(
        'Correlation',
        scale=alt.Scale(scheme='redblue', domain=[-1, 1]),
        legend=alt.Legend(title="Correlation")
    ),
    tooltip=['Variable 1', 'Variable 2', alt.Tooltip('Correlation', format='.2f')]
)

text = base.mark_text(size=8).encode(
    text='Correlation_Label',
    color=alt.condition(
        alt.expr.abs(alt.datum.Correlation) > 0.5,  #if absolute correlation is high
        alt.value('white'),                    #use white text
        alt.value('black')                     #else use black text
    )
)

chart = (heatmap + text).properties(
    title='Pearson correlation matrix (audio features, lyrics stats, metadata)',
    width=800,
    height=800
)

chart.interactive()

alt.LayerChart(...)

using a 0.9 threshold, then we could delete one of the columns between rolloff and zcr, and one of the columns between loudness and rms:
- rolloff will be kept over zcr, as the rolloff is a specific frequency below which a certain percentage of the total spectral energy lies, while zcr is the rate at which the signal changes sign (Higher ZCR means noisier or more percussive sounds), so zcr is more sensitive to noise.
- loudness will be kept over rms, as the loudness is measured in decibels (dB), which is a logarithmic scale that matches how human ears perceive volume changes. rms is linear, so its distribution can be skewed (loudness, being logarithmic, follows a more Gaussian distribution).